Audit ovarian GEO cohorts and clinical metadata

In [1]:
from pathlib import Path
import requests, gzip, re, json, hashlib, html
from datetime import datetime, timezone
from urllib.parse import urljoin

OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo'); OUT.mkdir(parents=True,exist_ok=True)
DATA=Path('/workspace/mechanism_fit_geo'); DATA.mkdir(parents=True,exist_ok=True)
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def save(name,obj):
    p=OUT/name; p.write_text(json.dumps(obj,indent=2,default=str)); return p

def family_url(acc):
    n=int(re.sub(r'\D','',acc)); return f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE{n//1000}nnn/{acc}/soft/{acc}_family.soft.gz'
def suppl_url(acc):
    n=int(re.sub(r'\D','',acc)); return f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE{n//1000}nnn/{acc}/suppl/'

def parse_soft(text):
    series={}; samples=[]; cur=None
    for line in text.splitlines():
        if line.startswith('^SERIES'):
            cur='series'; continue
        if line.startswith('^SAMPLE'):
            cur={'accession':line.split('=',1)[-1].strip()}; samples.append(cur); continue
        if not line.startswith('!'): continue
        k,v=(line[1:].split(' = ',1)+[''])[:2] if ' = ' in line[1:] else (line[1:],'')
        if isinstance(cur,dict):
            cur.setdefault(k,[]).append(v)
        elif cur=='series': series.setdefault(k,[]).append(v)
    return series,samples

def vals(sample, needles):
    out=[]
    for k,v in sample.items():
        if any(n in k.lower() for n in needles): out += v if isinstance(v,list) else [v]
    return out

# Literature-identified ovarian/HGSOC treatment-response or longitudinal candidates.
accessions=['GSE165897','GSE156699','GSE241908','GSE30161','GSE63885','GSE51373','GSE131978','GSE23554','GSE160626']
manifest={'lane':'Literature + GEO/ArrayExpress','started_at':ts(),'series':[]}
for acc in accessions:
    rec={'accession':acc,'access_timestamp':ts(),'soft_url':family_url(acc),'samples':[],'supplementary_files':[]}
    try:
        r=requests.get(rec['soft_url'],timeout=180)
        rec['soft_status']=r.status_code
        receipt={'url':rec['soft_url'],'timestamp':rec['access_timestamp'],'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:1000] if r.status_code!=200 else ''}
        save(f'{acc}_soft_access_receipt.json',receipt)
        if r.status_code==200:
            p=DATA/f'{acc}_family.soft.gz'; p.write_bytes(r.content); rec['soft_sha256']=sha(p); rec['soft_bytes']=p.stat().st_size
            text=gzip.decompress(r.content).decode('utf-8','replace'); series,samples=parse_soft(text)
            rec['series_metadata']={k:v for k,v in series.items() if any(x in k.lower() for x in ['title','summary','overall_design','relation','supplementary_file','pubmed','type','status'])}
            for s in samples:
                blob=' | '.join(sum([v if isinstance(v,list) else [str(v)] for v in s.values()],[]))
                patient=[]
                for pat in [r'patient(?: id)?[:= ]+([A-Za-z0-9_-]+)',r'\b(EOC\d+)\b',r'\b(OC\d+)\b']:
                    patient += re.findall(pat,blob,re.I)
                item={'accession':s.get('accession'),'title':vals(s,['title']),'source':vals(s,['source_name']),'characteristics':vals(s,['characteristics']),'treatment_protocol':vals(s,['treatment_protocol']),'relation':vals(s,['relation']),'patient_tokens':sorted(set(patient))}
                rec['samples'].append(item)
            # Directly supported unique patients only if explicit patient tokens exist; otherwise leave null.
            pts=sorted({p for s in rec['samples'] for p in s['patient_tokens']})
            rec['sample_count']=len(rec['samples']); rec['explicit_unique_patient_count']=len(pts) if pts else None; rec['explicit_patient_tokens']=pts
            alltext=json.dumps(rec['samples']).lower()
            rec['field_flags']={
                'pretreatment':any(k in alltext for k in ['pre-treatment','pretreatment','before chemotherapy','treatment-naïve','treatment naive','primary surgery']),
                'posttreatment':any(k in alltext for k in ['post-treatment','after chemotherapy','relapse','recurrent']),
                'response':any(k in alltext for k in ['response','responder','sensitive','resistant','recist','complete response','progressive disease']),
                'progression_pfi':any(k in alltext for k in ['progression','pfi','pfs','platinum-free interval']),
                'survival_followup':any(k in alltext for k in ['overall survival','follow-up','follow up','censor']),
                'treatment_line_or_index':any(k in alltext for k in ['first-line','second-line','neoadjuvant','adjuvant','cycle','course','line of therapy'])}
            save(f'{acc}_parsed_metadata.json',rec)
        # Enumerate supplementary listing and download small processed/clinical files.
        surl=suppl_url(acc); sr=requests.get(surl,timeout=90); rec['supplementary_listing_status']=sr.status_code
        save(f'{acc}_supplementary_listing_receipt.json',{'url':surl,'timestamp':ts(),'status_code':sr.status_code,'headers':{k:v for k,v in sr.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':sr.text[:5000]})
        if sr.status_code==200:
            hrefs=re.findall(r'href="([^"]+)"',sr.text,re.I)
            for href in hrefs:
                if href.startswith('?') or href.startswith('/') or href in ['../']: continue
                url=urljoin(surl,html.unescape(href)); name=Path(html.unescape(href)).name
                if not name: continue
                fe={'name':name,'url':url,'access_timestamp':ts()}
                # HEAD size for auditable enumeration.
                try:
                    hr=requests.head(url,timeout=60,allow_redirects=True); fe['status_code']=hr.status_code; fe['declared_size']=int(hr.headers.get('content-length','0') or 0); fe['content_type']=hr.headers.get('content-type')
                    low=name.lower()
                    if hr.status_code==200 and fe['declared_size']<=100_000_000 and any(x in low for x in ['meta','clinical','phenotype','sample','matrix','count','expr','data','.csv','.tsv','.txt','.xlsx','.xls']):
                        dr=requests.get(url,timeout=300); fe['download_status']=dr.status_code
                        if dr.status_code==200:
                            p=DATA/f'{acc}__{name}'; p.write_bytes(dr.content); fe.update({'local_path':str(p),'bytes':p.stat().st_size,'sha256':sha(p)})
                except Exception as e: fe['error']=type(e).__name__+': '+str(e)[:300]
                rec['supplementary_files'].append(fe)
    except Exception as e:
        rec['error']=type(e).__name__+': '+str(e)[:500]
    manifest['series'].append(rec)

manifest['completed_at']=ts(); save('lane4_manifest.json',manifest)
print(json.dumps({'series':len(manifest['series']),'soft_accessed':sum(1 for x in manifest['series'] if x.get('soft_status')==200),'samples':sum(x.get('sample_count',0) for x in manifest['series']),'downloaded_supplements':sum(sum(1 for f in x.get('supplementary_files',[]) if f.get('sha256')) for x in manifest['series']),'output':str(OUT)},indent=2))

{
  "series": 9,
  "soft_accessed": 9,
  "samples": 403,
  "downloaded_supplements": 11,
  "output": "/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo"
}


Verify GSE30161 matrix and complete-case outcomes

In [3]:
from pathlib import Path
import requests, gzip, hashlib, json, re, pandas as pd
from datetime import datetime, timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo'); DATA=Path('/workspace/mechanism_fit_geo')
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
 return h.hexdigest()
acc='GSE30161'; url='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE30nnn/GSE30161/matrix/GSE30161_series_matrix.txt.gz'
r=requests.get(url,timeout=300); rec={'url':url,'timestamp':ts(),'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:1000] if r.status_code!=200 else ''}
(OUT/'GSE30161_series_matrix_access_receipt.json').write_text(json.dumps(rec,indent=2))
result={'access':rec}
if r.status_code==200:
 p=DATA/'GSE30161_series_matrix.txt.gz'; p.write_bytes(r.content); result.update({'path':str(p),'bytes':p.stat().st_size,'sha256':sha(p)})
 # Parse matrix header annotations and sample table.
 text=gzip.decompress(r.content).decode('utf-8','replace'); lines=text.splitlines(); anns={}
 for line in lines:
  if line.startswith('!Sample_'):
   parts=line.split('\t'); anns[parts[0][1:]]=[x.strip('"') for x in parts[1:]]
 result['sample_annotation_keys']=sorted(anns)
 result['sample_count']=len(anns.get('Sample_geo_accession',[]))
 # Family SOFT clinical complete-case counts.
 d=json.loads((OUT/'GSE30161_parsed_metadata.json').read_text())
 rows=[]
 for s in d['samples']:
  kv={}
  for x in s['characteristics']:
   if ':' in x:
    k,v=x.split(':',1); kv[k.strip().lower()]=v.strip()
  kv['geo_accession']=s['accession']; kv['title']=s['title'][0] if s['title'] else ''
  rows.append(kv)
 df=pd.DataFrame(rows)
 required=['histo','stage','surgdate','chemtype','chemoresponse','chemo agent','date.last.follow.up','pfi days','relapse(1=yes, 0=no)','overall survival days','censoring(dead=1,alive=0)']
 result['clinical_columns']=list(df.columns); result['required_columns']=required
 result['complete_case_count']=int(df[required].replace('',pd.NA).notna().all(axis=1).sum())
 result['unique_geo_accessions']=int(df.geo_accession.nunique()); result['unique_titles']=int(df.title.nunique())
 result['response_counts']=df['chemoresponse'].value_counts(dropna=False).to_dict(); result['relapse_counts']=df['relapse(1=yes, 0=no)'].value_counts(dropna=False).to_dict(); result['censoring_counts']=df['censoring(dead=1,alive=0)'].value_counts(dropna=False).to_dict()
 outcsv=DATA/'GSE30161_clinical_fields_audit.csv'; df.to_csv(outcsv,index=False); result['clinical_audit_sha256']=sha(outcsv)
(OUT/'GSE30161_matrix_and_complete_case_audit.json').write_text(json.dumps(result,indent=2,default=str))
print(json.dumps(result,indent=2,default=str))

{
  "access": {
    "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE30nnn/GSE30161/matrix/GSE30161_series_matrix.txt.gz",
    "timestamp": "2026-07-16T20:57:14.441103Z",
    "status_code": 200,
    "headers": {
      "Last-Modified": "Sun, 05 Jul 2026 10:05:00 GMT",
      "Content-Length": "9004224",
      "Content-Type": "application/x-gzip"
    },
    "body": ""
  },
  "path": "/workspace/mechanism_fit_geo/GSE30161_series_matrix.txt.gz",
  "bytes": 9004224,
  "sha256": "62eee95645965d402723a9e956887fd106482a1ede5dce71efdf9d743d55722c",
  "sample_annotation_keys": [
    "Sample_channel_count",
    "Sample_characteristics_ch1",
    "Sample_contact_address",
    "Sample_contact_city",
    "Sample_contact_country",
    "Sample_contact_email",
    "Sample_contact_institute",
    "Sample_contact_name",
    "Sample_contact_zip/postal_code",
    "Sample_data_processing",
    "Sample_data_row_count",
    "Sample_description",
    "Sample_extract_protocol_ch1",
    "Sample_geo_accession",
 

Verify GSE63885 processed matrix and outcomes

In [5]:
from pathlib import Path
import requests,gzip,hashlib,json,pandas as pd,re
from datetime import datetime,timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo'); DATA=Path('/workspace/mechanism_fit_geo')
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
 return h.hexdigest()
acc='GSE63885'; url='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn/GSE63885/matrix/GSE63885_series_matrix.txt.gz'; r=requests.get(url,timeout=300)
rec={'url':url,'timestamp':ts(),'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:1000] if r.status_code!=200 else ''}; (OUT/f'{acc}_series_matrix_access_receipt.json').write_text(json.dumps(rec,indent=2))
res={'access':rec}
if r.status_code==200:
 p=DATA/f'{acc}_series_matrix.txt.gz'; p.write_bytes(r.content); res.update({'bytes':p.stat().st_size,'sha256':sha(p)})
 text=gzip.decompress(r.content).decode('utf-8','replace'); res['sample_count']=sum(1 for l in text.splitlines() if l.startswith('!Sample_geo_accession')) and len(next(l for l in text.splitlines() if l.startswith('!Sample_geo_accession')).split('\t'))-1
 d=json.loads((OUT/f'{acc}_parsed_metadata.json').read_text()); rows=[]
 for s in d['samples']:
  kv={}
  for x in s['characteristics']:
   if ':' in x:
    k,v=x.split(':',1);kv[k.strip().lower()]=v.strip()
  kv['geo_accession']=s['accession'];rows.append(kv)
 df=pd.DataFrame(rows); unknown=lambda s:s.astype(str).str.strip().str.lower().isin(['na','unknown','unkown','n/a',''])
 cols={
  'response':'clinical status post 1st line chemotherapy (cr - complete response; pr - partial response; sd - stable disease; p - progression)',
  'last_status':'clinical status at last follow-up (awd - alive with disease; dod - dead from disease; ned - no evidence of disease)',
  'dfs':'dfs - disease-free survival [days]','os':'os - overall survival [days]','chemo':'adjuwant chemotherapy','histology':'histophatological type of tumor','stage':'figo_stage'}
 valid=df[cols['response']].isin(['CR','PR','SD','P']) & (~unknown(df[cols['last_status']])) & (~unknown(df[cols['dfs']])) & df[cols['dfs']].astype(str).str.fullmatch(r'\d+(\.\d+)?') & (~unknown(df[cols['os']])) & df[cols['os']].astype(str).str.fullmatch(r'\d+(\.\d+)?') & (~unknown(df[cols['chemo']])) & (~unknown(df[cols['histology']])) & (~unknown(df[cols['stage']]))
 res['clinical_columns']=list(df.columns);res['explicit_complete_count']=int(valid.sum());res['n_rows']=len(df);res['counts']={k:df[v].value_counts(dropna=False).to_dict() for k,v in cols.items()}
(OUT/f'{acc}_matrix_and_complete_case_audit.json').write_text(json.dumps(res,indent=2,default=str))
print(json.dumps(res,indent=2,default=str))

{
  "access": {
    "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE63nnn/GSE63885/matrix/GSE63885_series_matrix.txt.gz",
    "timestamp": "2026-07-16T21:03:47.595495Z",
    "status_code": 200,
    "headers": {
      "Last-Modified": "Wed, 17 Jun 2026 06:44:30 GMT",
      "Content-Length": "30104516",
      "Content-Type": "application/x-gzip"
    },
    "body": ""
  },
  "bytes": 30104516,
  "sha256": "9815d81e390dd2e429c5857cb35da57a072118ce68bdea21282c2db130b15f94",
  "sample_count": 101,
  "clinical_columns": [
    "clinical sample",
    "histophatological type of tumor",
    "adjuwant chemotherapy",
    "brca1 mutation",
    "somatic tp53 mutation",
    "tp53 accumulation in cancer cells (assesed by immunohistochemistry)",
    "tumor grade (g1 - well differentiated; g2 - moderately differentiated; g3 - poorly differentiated; g4 - undifferentiated)",
    "figo_stage",
    "residual tumor size (r0 < 1 cm;  1 < r1 < 5 cm; r2 > 5cm)",
    "clinical status post 1st line chemotherapy

Cross-review mechanism relevance benchmark eligibility

In [7]:
from pathlib import Path
import json
from datetime import datetime,timezone
root=Path('/mnt/shared-workspace/shared/mechanism_fit_audit'); m=json.loads((root/'lane5_internal/lane5_manifest.json').read_text())
review={'reviewer_lane':'Literature worker cross-review','timestamp':datetime.now(timezone.utc).isoformat().replace('+00:00','Z'),'qualifying_independent_truth_count':sum(x['qualifies_as_independent_truth'] for x in m['reviewer_artifacts']),'real_source_packet_status':'No provenance-complete raw case packet was verified; two JSONs are engine-derived traces and one image lacks source provenance.','canonical_corpus_check':{'observed_rows':m['canonical_corpus']['table_counts'].get('trials'),'observed_sha256':m['canonical_corpus']['sha256'],'frozen_claim_rows':3195,'frozen_claim_sha256':'c2c248c975dc3dd776ba2cc8b78a7a2eb97b06fe6e9c11bfcd018593468b97a8','matches_frozen':False},'negative_refusal_controls':'No independent observed negative/refusal case labels were verified.','verdict':'INSUFFICIENT','exact_missing_field':'Named qualified independent blinded case-to-trial labels with attestation; provenance-complete case packets; observed negative/refusal controls; matching frozen trial corpus artifact.'}
(root/'cross_review_mechanism.json').write_text(json.dumps(review,indent=2));print(json.dumps(review,indent=2))

{
  "reviewer_lane": "Literature worker cross-review",
  "timestamp": "2026-07-16T21:05:00.312818Z",
  "qualifying_independent_truth_count": 0,
  "real_source_packet_status": "No provenance-complete raw case packet was verified; two JSONs are engine-derived traces and one image lacks source provenance.",
  "canonical_corpus_check": {
    "observed_rows": 2897,
    "observed_sha256": "a9c967270dd0e188eb113a0dcbca49a2dd33c118ee905f30331b8a5d9e4a9926",
    "frozen_claim_rows": 3195,
    "frozen_claim_sha256": "c2c248c975dc3dd776ba2cc8b78a7a2eb97b06fe6e9c11bfcd018593468b97a8",
    "matches_frozen": false
  },
  "negative_refusal_controls": "No independent observed negative/refusal case labels were verified.",
  "verdict": "INSUFFICIENT",
  "exact_missing_field": "Named qualified independent blinded case-to-trial labels with attestation; provenance-complete case packets; observed negative/refusal controls; matching frozen trial corpus artifact."
}


Search ArrayExpress for ovarian outcome cohorts

In [9]:
from pathlib import Path
import requests,json,hashlib
from datetime import datetime,timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo')
def ts():return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
queries=['ovarian chemotherapy response','high grade serous ovarian platinum resistance','ovarian progression free survival transcriptome']
base='https://www.ebi.ac.uk/biostudies/api/v1/search'
allhits=[]; attempts=[]
for q in queries:
 params={'query':q,'pageSize':100,'collection':'ArrayExpress'}; at=ts(); r=requests.get(base,params=params,timeout=90)
 rec={'url':r.url,'timestamp':at,'status_code':r.status_code,'headers':{k:v for k,v in r.headers.items() if k.lower() in ['content-type','content-length','etag','last-modified']},'body':r.text[:10000]}
 attempts.append(rec)
 if r.status_code==200:
  try:
   obj=r.json(); hits=obj.get('hits') or obj.get('entries') or obj.get('results') or []
   for h in hits:
    h['_query']=q; allhits.append(h)
  except Exception: pass
# Deduplicate and retain exact response evidence.
seen=set(); uniq=[]
for h in allhits:
 key=str(h.get('accno') or h.get('accession') or h.get('id') or h)
 if key not in seen:seen.add(key);uniq.append(h)
out={'queries':queries,'attempts':attempts,'unique_hits':uniq,'timestamp':ts()}
p=OUT/'arrayexpress_biostudies_search_receipt.json';p.write_text(json.dumps(out,indent=2,default=str))
print(json.dumps({'attempts':[(x['status_code'],x['url']) for x in attempts],'unique_hit_count':len(uniq),'hits':[(h.get('accno') or h.get('accession') or h.get('id'),h.get('title')) for h in uniq[:50]],'receipt_sha256':hashlib.sha256(p.read_bytes()).hexdigest()},indent=2))

{
  "attempts": [
    [
      200,
      "https://www.ebi.ac.uk/biostudies/api/v1/search?query=ovarian+chemotherapy+response&pageSize=100&collection=ArrayExpress"
    ],
    [
      200,
      "https://www.ebi.ac.uk/biostudies/api/v1/search?query=high+grade+serous+ovarian+platinum+resistance&pageSize=100&collection=ArrayExpress"
    ],
    [
      200,
      "https://www.ebi.ac.uk/biostudies/api/v1/search?query=ovarian+progression+free+survival+transcriptome&pageSize=100&collection=ArrayExpress"
    ]
  ],
  "unique_hit_count": 268,
  "hits": [
    [
      "E-GEOD-23616",
      "Integrated Genomics of Ovarian Xenograft Tumor Progression and Chemotherapy Response"
    ],
    [
      "E-GEOD-30161",
      "Genomic Multivariate Predictors of Response to Adjuvant Chemotherapy in Ovarian Carcinoma: Predicting Platinum Resistance"
    ],
    [
      "E-GEOD-7463",
      "Transcription profiling of human ovarian tumors (43) from untreated patients prior to surgery vs. chemotherapy treated pri

Inspect top nonduplicate ArrayExpress ovarian cohorts

In [11]:
from pathlib import Path
import requests,gzip,re,json,hashlib
from datetime import datetime,timezone
OUT=Path('/mnt/shared-workspace/shared/mechanism_fit_audit/lane4_geo'); DATA=Path('/workspace/mechanism_fit_geo')
def ts(): return datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def parse(text):
 series={}; samples=[]; cur=None
 for line in text.splitlines():
  if line.startswith('^SERIES'):cur='series';continue
  if line.startswith('^SAMPLE'):cur={'accession':line.split('=',1)[-1].strip()};samples.append(cur);continue
  if not line.startswith('!') or ' = ' not in line:continue
  k,v=line[1:].split(' = ',1)
  if isinstance(cur,dict):cur.setdefault(k,[]).append(v)
  elif cur=='series':series.setdefault(k,[]).append(v)
 return series,samples
res=[]
for acc in ['GSE43867','GSE25203','GSE31245']:
 n=int(acc[3:]);url=f'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE{n//1000}nnn/{acc}/soft/{acc}_family.soft.gz';at=ts();r=requests.get(url,timeout=180)
 rec={'accession':acc,'url':url,'timestamp':at,'status_code':r.status_code}
 if r.status_code==200:
  p=DATA/f'{acc}_family.soft.gz';p.write_bytes(r.content);rec.update({'bytes':p.stat().st_size,'sha256':sha(p)})
  series,samples=parse(gzip.decompress(r.content).decode('utf-8','replace')); rec['series']={k:v for k,v in series.items() if any(x in k.lower() for x in ['title','summary','overall_design','pubmed','type'])};rec['sample_count']=len(samples)
  fields={}
  for s in samples:
   for k,vs in s.items():
    if 'characteristics' in k.lower():
     for x in vs:
      if ':' in x:
       kk,v=x.split(':',1); fields.setdefault(kk.strip().lower(),set()).add(v.strip())
  rec['clinical_fields']={k:sorted(v)[:100] for k,v in fields.items()};rec['clinical_field_value_counts']={k:len(v) for k,v in fields.items()}
 (OUT/f'{acc}_arrayexpress_candidate_receipt.json').write_text(json.dumps(rec,indent=2,default=str));res.append(rec)
print(json.dumps([{k:v for k,v in x.items() if k not in ['clinical_fields']}|{'clinical_field_names':sorted(x.get('clinical_fields',{})),'selected_values':{k:v for k,v in x.get('clinical_fields',{}).items() if any(q in k for q in ['response','survival','pfi','progress','follow','treat','stage','patient','hist'])}} for x in res],indent=2,default=str))

[
  {
    "accession": "GSE43867",
    "url": "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE43nnn/GSE43867/soft/GSE43867_family.soft.gz",
    "timestamp": "2026-07-16T21:07:23.673335Z",
    "status_code": 200,
    "bytes": 342491,
    "sha256": "7b0021868f9a6867cf1bbc02c9bb8f289f8a455194c88835c7e768821a4c50d5",
    "series": {
      "Series_title": [
        "microRNA expression for 86 chemo-treated ovarian cancer patients"
      ],
      "Series_pubmed_id": [
        "23697367"
      ],
      "Series_summary": [
        "We profile expression in serous epithelial ovarian carcinomas to assess the possibility of an miRNA signature associated with chemoresistance.",
        "",
        "Resistance to the available therapies is one of the main causes of low survival of patients with advanced epithelial ovarian cancer, thus representing an emergency in oncology. Here, we profiled miRNA expression in 86 na\u00efve serous epithelial ovarian carcinomas (EOCs) to assess the possibility of miRNAs